# Complete Deep Learning Tutorial

This notebook is a practical beginner-to-intermediate guide to Deep Learning. It explains the core ideas behind neural networks and gives runnable examples with TensorFlow/Keras.

**How to use this notebook**

1. Run the setup cells first.
2. Read one section at a time.
3. Change small parts of the code and rerun it.
4. Watch training curves, not only final accuracy.
5. Write notes about overfitting, loss, accuracy, and model mistakes.

**Main library used here**: TensorFlow/Keras. If TensorFlow is not installed, run `pip install tensorflow` in your environment.

## 1. What Is Deep Learning?

Deep Learning is a part of Machine Learning that uses neural networks with many layers. These layers learn useful representations from data.

Deep learning is especially strong for:

- Images and computer vision
- Text and natural language processing
- Audio and speech
- Large datasets with complex patterns

For small tabular datasets, classical ML models like Random Forest, XGBoost, and Logistic Regression are often better starting points.

## 2. Environment Setup

Install the main packages if needed:

```bash
pip install tensorflow numpy pandas matplotlib scikit-learn jupyter
```

Optional later:

```bash
pip install torch torchvision transformers datasets
```

This notebook uses TensorFlow/Keras because it is friendly for beginners and has clean high-level APIs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print("TensorFlow version:", tf.__version__)
except ImportError:
    print("TensorFlow is not installed. Run: pip install tensorflow")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 3. Core Vocabulary

| Term | Meaning |
|---|---|
| Tensor | Multidimensional array, like a vector, matrix, or image batch |
| Neuron | Small computation unit that applies weights, bias, and activation |
| Layer | Group of neurons that transforms input data |
| Activation | Nonlinear function such as ReLU, sigmoid, or softmax |
| Loss | Error value the model tries to reduce |
| Optimizer | Algorithm that updates weights, such as Adam or SGD |
| Epoch | One full pass over the training data |
| Batch | Small group of examples processed together |
| Backpropagation | Method for calculating gradients through the network |

## 4. A Neuron Is a Weighted Sum

A basic neuron calculates:

```text
output = activation(w1*x1 + w2*x2 + ... + b)
```

Weights and bias are learned from data. The activation function lets the network learn nonlinear patterns.

In [ ]:
x = np.array([2.0, 3.0])
w = np.array([0.4, -0.7])
b = 0.2

z = np.dot(x, w) + b
relu_output = max(0, z)

print("Weighted sum:", z)
print("After ReLU:", relu_output)

## 5. First Neural Network: Binary Classification

We will create a small curved dataset and train a neural network to separate two classes. This is a good first example because a simple straight line is not enough.

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15, alpha=0.8)
plt.title("Two Moons Classification Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0,
)

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
def plot_history(history):
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    history_df[["loss", "val_loss"]].plot(ax=axes[0])
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")

    history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1])
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")

    plt.show()

plot_history(history)

## 6. Activation Functions

| Activation | Common Use |
|---|---|
| ReLU | Hidden layers for most neural networks |
| Sigmoid | Binary classification output |
| Softmax | Multiclass classification output |
| Tanh | Sometimes used in recurrent networks |

Without nonlinear activation functions, many layers collapse into something similar to one linear model.

In [ ]:
x_values = np.linspace(-5, 5, 200)
relu = np.maximum(0, x_values)
sigmoid = 1 / (1 + np.exp(-x_values))
tanh = np.tanh(x_values)

plt.figure(figsize=(8, 5))
plt.plot(x_values, relu, label="ReLU")
plt.plot(x_values, sigmoid, label="Sigmoid")
plt.plot(x_values, tanh, label="Tanh")
plt.legend()
plt.title("Common Activation Functions")
plt.grid(True)
plt.show()

## 7. Loss Functions and Output Layers

| Problem | Output Layer | Loss |
|---|---|---|
| Binary classification | `Dense(1, activation="sigmoid")` | `binary_crossentropy` |
| Multiclass classification | `Dense(num_classes, activation="softmax")` | `sparse_categorical_crossentropy` or `categorical_crossentropy` |
| Regression | `Dense(1)` | `mse`, `mae`, or Huber loss |

Pick the output layer and loss function together. A mismatch here is a common beginner bug.

## 8. MNIST Digit Classifier

MNIST is a classic dataset of handwritten digits from 0 to 9. Each image is 28x28 pixels.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
mnist_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax"),
])

mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

mnist_history = mnist_model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1,
)

In [ ]:
test_loss, test_accuracy = mnist_model.evaluate(X_test, y_test, verbose=0)
print(f"MNIST test accuracy: {test_accuracy:.4f}")

pred_probs = mnist_model.predict(X_test[:10], verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)
print("Predictions:", pred_labels)
print("Actual     :", y_test[:10])

## 9. Convolutional Neural Networks

CNNs are designed for images. They learn filters that detect simple patterns first, then combine them into more complex patterns.

Important layers:

- `Conv2D`: learns image filters
- `MaxPooling2D`: reduces image size while keeping strong signals
- `Flatten`: converts feature maps into a vector
- `Dense`: performs final classification

In [ ]:
X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.summary()

Training a CNN can take longer than the dense model. Start with 1 or 2 epochs if your computer is slow.

In [ ]:
# Uncomment to train the CNN.
# cnn_history = cnn_model.fit(
#     X_train_cnn,
#     y_train,
#     validation_split=0.1,
#     epochs=2,
#     batch_size=128,
# )
# cnn_model.evaluate(X_test_cnn, y_test)

## 10. Overfitting and Regularization

Overfitting happens when the model memorizes training data and performs worse on new data.

Common fixes:

- Add more data
- Use data augmentation
- Make the model smaller
- Add dropout
- Add L2 regularization
- Use early stopping
- Train for fewer epochs

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

regularized_model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

regularized_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

regularized_history = regularized_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0,
)

print("Epochs actually trained:", len(regularized_history.history["loss"]))

## 11. Transfer Learning

Transfer learning means starting from a model trained on a large dataset and adapting it to your task.

This is common in computer vision and NLP because large pretrained models already know useful patterns.

Typical steps:

1. Load a pretrained base model.
2. Freeze its layers.
3. Add your own classification head.
4. Train the head on your dataset.
5. Optionally unfreeze some layers and fine-tune slowly.

In [ ]:
# Example architecture only. This may download pretrained weights if weights="imagenet".
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None,
)
base_model.trainable = False

transfer_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation="sigmoid"),
])

transfer_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
transfer_model.summary()

## 12. Text Classification Overview

Deep learning for text usually follows this path:

1. Clean or normalize text.
2. Tokenize text into words or subwords.
3. Convert tokens to numbers.
4. Learn embeddings or use pretrained embeddings.
5. Train a model such as CNN, RNN, LSTM, GRU, or Transformer.

For beginners, start with TF-IDF plus Logistic Regression or Naive Bayes before deep learning. It is fast and surprisingly strong.

In [ ]:
texts = np.array([
    "this movie is excellent",
    "i loved the acting",
    "what a boring film",
    "the story was terrible",
    "amazing and emotional",
    "bad acting and slow story",
])
labels = np.array([1, 1, 0, 0, 1, 0])

vectorizer = layers.TextVectorization(max_tokens=1000, output_sequence_length=8)
vectorizer.adapt(texts)

text_model = keras.Sequential([
    vectorizer,
    layers.Embedding(input_dim=1000, output_dim=16),
    layers.GlobalAveragePooling1D(),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

text_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
text_model.fit(texts, labels, epochs=20, verbose=0)

sample_text = np.array(["excellent acting and amazing story"])
print("Positive probability:", float(text_model.predict(sample_text, verbose=0)[0][0]))

## 13. Practical Deep Learning Workflow

Use this checklist for real projects:

1. Define the problem and metric.
2. Start with a simple baseline.
3. Split train, validation, and test data correctly.
4. Normalize numeric input data.
5. Choose output layer and loss function correctly.
6. Train a small model first.
7. Plot loss and metric curves.
8. Check overfitting and underfitting.
9. Improve with architecture, data, augmentation, or regularization.
10. Evaluate on the untouched test set.
11. Inspect wrong predictions.
12. Save the model and document limitations.

## 14. Saving and Loading a Model

Save models when you want to reuse them later for prediction or deployment.

In [ ]:
mnist_model.save("mnist_dense_model.keras")
loaded_model = keras.models.load_model("mnist_dense_model.keras")

loaded_loss, loaded_accuracy = loaded_model.evaluate(X_test, y_test, verbose=0)
print(f"Loaded model accuracy: {loaded_accuracy:.4f}")

## 15. Mini Projects

Choose one project and complete it in a separate notebook:

| Project | Skills |
|---|---|
| MNIST digit classifier | Dense network, CNN, metrics |
| Fashion MNIST classifier | Image classification, error analysis |
| Cat vs dog classifier | CNN, data loading, augmentation |
| Movie review sentiment | Text vectorization, embeddings |
| Customer churn neural network | Tabular preprocessing, binary classification |

For every project, include problem statement, dataset, model architecture, training curves, final metric, wrong predictions, and next improvements.

## 16. Best Resources

Use these in order:

1. TensorFlow Tutorials: https://www.tensorflow.org/tutorials
2. Keras Guides: https://keras.io/guides/
3. fast.ai Practical Deep Learning for Coders: https://course.fast.ai/
4. PyTorch Learn the Basics: https://docs.pytorch.org/tutorials/beginner/basics/intro.html
5. 3Blue1Brown Neural Networks: https://www.3blue1brown.com/topics/neural-networks
6. Hugging Face Course: https://huggingface.co/learn/nlp-course

Suggested learning order:

1. Neural network basics
2. Dense networks with Keras
3. Training curves and overfitting
4. CNNs for images
5. Transfer learning
6. Text vectorization and embeddings
7. Transformers after you understand the basics

## 17. Revision Checklist

You should be able to explain:

- What is a tensor?
- What are weights and bias?
- Why do neural networks need activation functions?
- What is the difference between loss and metric?
- What does an optimizer do?
- What is backpropagation?
- What are epochs and batch size?
- Why does overfitting happen?
- How do dropout and early stopping help?
- Why are CNNs good for images?
- What is transfer learning?
- How do embeddings represent text?

Final rule: do not only chase accuracy. Understand the data, the metric, the mistakes, and why the model behaves the way it does.